# Reading H1 Receiver Observation Data

How to read the HDF5 files the H1 Receiver scheduler produces, inspect the observation
metadata, and plot a waterfall and a mean spectrum.

## Two dataset names, and the name is the units

A recording holds its spectra under **one of two names**:

| dataset | units | means |
|---|---|---|
| `spectra_kelvin` | K | the bandpass and gain in force were applied when the file was written |
| `spectra_linear` | counts | no calibration applied to that tuning, so the data are raw |

Asking for the wrong name raises `KeyError` rather than quietly handing back numbers on
the other scale — which is the point, and is why the units are not merely an attribute
you could forget to read.

Calibration is applied at write time but never lost: `bandpass_correction`,
`applied_gain_counts_per_k` and `applied_t_sys_k` all travel in the file, so raw counts
are one line away (section 2). Kelvin is already linear in the quantity of interest, so
it is plotted directly; only counts get converted to dB for display.

## Filenames

`data/observations/YYYYMMDD_HHMMSS_<mode>.h5`, where mode is `track` (the mount followed
the sky), `drift` (the dish was parked and the sky moved through the beam) or `manual`
(started at the console). Everything else about the observation is inside the file.

In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

ModuleNotFoundError: No module named 'h5py'

## 1. Select and open an HDF5 file

List available `.h5` files in the data folder and pick one to examine.

In [ ]:
# Recordings live in their own folder. The rest of data/ is plots, horizon
# partials and gain fits, so globbing data/ used to return mostly non-observations.
data_dir = Path('data') / 'observations'
folder = data_dir if data_dir.exists() else Path('.')

# By modification time, so FILE_INDEX = -1 really is the most recent. The
# current names sort chronologically anyway - that is the point of leading with
# the timestamp - but recordings made before 2026-08-25 start with "h1_" and
# would sort after every one of them.
h5_files = sorted(folder.glob('*.h5'), key=lambda p: p.stat().st_mtime)

print(f'Found {len(h5_files)} recording(s) in {folder}:')
for i, f in enumerate(h5_files):
    print(f'  [{i}] {f.name}')

# Select which file to open (change this index as needed)
FILE_INDEX = -1  # -1 = most recent
h5_path = h5_files[FILE_INDEX]
print(f'\nSelected: {h5_path}')

## 2. Read metadata and datasets

In [ ]:
# A file the receiver is still writing needs swmr=True (the plain open hits
# the lock, which is right for a file not written that way). Try plain, then live.
try:
    hf = h5py.File(h5_path, 'r')
except OSError:
    hf = h5py.File(h5_path, 'r', swmr=True)
    print('(reading a recording in progress: what has arrived so far)')

freq_hz = hf['frequency_hz'][:]
freq_mhz = freq_hz / 1e6
timestamps = hf['timestamps'][:]
integration_times = hf['integration_times'][:]

# The dataset name is the units. Do not guess, and do not fall back to the
# other one: a file recorded in kelvin and read as if it were counts gives
# numbers that are wrong by the gain and offset by T_sys, and looks fine.
calibrated = 'spectra_kelvin' in hf
spectra = hf['spectra_kelvin' if calibrated else 'spectra_linear'][:]
units = 'K' if calibrated else 'counts'

n_spectra, n_channels = spectra.shape
attrs = dict(hf.attrs)

print(f'Datasets loaded: {n_spectra} spectra x {n_channels} channels, in {units}')
print(f'Frequency range: {freq_mhz[0]:.3f} - {freq_mhz[-1]:.3f} MHz')
print(f'Time span: {timestamps[-1] - timestamps[0]:.1f} seconds')
print(f'Total integration: {np.sum(integration_times):.1f} seconds')

# The calibration travels with the file, so it is exactly reversible. Uncomment
# to recover the raw counts the SDR actually produced - needed if you want to
# re-reduce with a better bandpass or a better gain than the one in force at
# the time.
# if calibrated:
#     raw_counts = ((spectra + attrs['applied_t_sys_k'])
#                   * attrs['applied_gain_counts_per_k']
#                   * hf['bandpass_correction'][:])

if calibrated:
    print(f"Calibration applied: gain {attrs['applied_gain_counts_per_k']:.4g} counts/K, "
          f"T_sys {attrs['applied_t_sys_k']:.1f} K")
    valid = hf['bandpass_valid'][:]
    if not valid.all():
        print(f'  note: {(~valid).sum()} channels fell outside the bandpass template '
              f'and were left uncorrected')

## 3. Display observation metadata

Show all attributes stored in the HDF5 file, including receiver settings and
observation context from the scheduler.

In [ ]:
# Receiver metadata (always present)
print('=== Receiver Settings ===')
print(f"  SDR type:          {attrs.get('sdr_type', 'N/A')}")
print(f"  Center frequency:  {attrs.get('center_freq_hz', 0) / 1e6:.3f} MHz")
print(f"  Sample rate:       {attrs.get('sample_rate_hz', 0) / 1e6:.3f} MHz")
print(f"  FFT size:          {attrs.get('fft_size', 'N/A')}")
print(f"  Gain:              {attrs.get('gain_db', 'N/A')} dB")
print(f"  Integration time:  {attrs.get('nominal_integration_time', 'N/A')} s")
print(f"  Created:           {attrs.get('created', 'N/A')}")
print(f"  Spectra units:     {attrs.get('spectra_units', units)}")
if attrs.get('segment'):
    print(f"  Segment:           {attrs['segment']} (previous file ended: "
          f"{attrs.get('segment_reason', '?')} changed)")

# Observation metadata (present when launched from scheduler)
if 'obs_name' in attrs:
    print()
    print('=== Observation Context ===')
    print(f"  Name:              {attrs.get('obs_name')}")
    if attrs.get('comment'):
        print(f"  Comment:           {attrs.get('comment')}")
    # track = the mount followed the sky; drift = it was parked and the sky
    # moved through the beam. This decides whether successive rows of the
    # waterfall are the same piece of sky, so read it before averaging them.
    print(f"  Mode:              {attrs.get('observation_mode', 'N/A')}")
    print(f"  Scheduled:         {attrs.get('start_date', '')} {attrs.get('start_time', '')}")
    print(f"  Duration:          {attrs.get('duration_minutes', 'N/A')} minutes")
    print(f"  Calibrator:        {'ON' if attrs.get('calibrator', 0) else 'OFF'}")

    coord_sys = attrs.get('coord_system', '')
    if coord_sys == 'object':
        print(f"  Target:            {attrs.get('object_name', 'N/A')} (solar system object)")
    elif coord_sys == 'radec':
        ra = attrs.get('coord1_deg', 0) + attrs.get('coord1_min', 0)/60 + attrs.get('coord1_sec', 0)/3600
        dec = attrs.get('coord2_deg', 0) + attrs.get('coord2_min', 0)/60 + attrs.get('coord2_sec', 0)/3600
        print(f"  Target:            RA {ra:.4f}h, Dec {dec:.4f}°")
    elif coord_sys == 'galactic':
        l = attrs.get('coord1_deg', 0) + attrs.get('coord1_min', 0)/60 + attrs.get('coord1_sec', 0)/3600
        b = attrs.get('coord2_deg', 0) + attrs.get('coord2_min', 0)/60 + attrs.get('coord2_sec', 0)/3600
        print(f"  Target:            Gal l={l:.2f}°, b={b:.2f}°")
    elif coord_sys == 'altaz':
        alt = attrs.get('coord1_deg', 0) + attrs.get('coord1_min', 0)/60 + attrs.get('coord1_sec', 0)/3600
        az = attrs.get('coord2_deg', 0) + attrs.get('coord2_min', 0)/60 + attrs.get('coord2_sec', 0)/3600
        print(f"  Target:            Alt {alt:.2f}°, Az {az:.2f}°")
else:
    print('\n  (No scheduler metadata - receiver was run standalone)')

## 4. Build a descriptive title from metadata

Used by the plots below.

In [ ]:
def make_title():
    """Build a plot title from observation metadata."""
    parts = []
    if 'obs_name' in attrs:
        parts.append(str(attrs['obs_name']))
    if attrs.get('coord_system') == 'object':
        parts.append(str(attrs.get('object_name', '')).capitalize())
    if attrs.get('observation_mode'):
        parts.append(str(attrs['observation_mode']))
    if attrs.get('calibrator', 0):
        parts.append('(CAL ON)')
    parts.append(str(attrs.get('created', '')[:19]))
    return ' — '.join(parts) if parts else h5_path.name

# What to plot, and what to call it. Counts span orders of magnitude across the
# band and want a log scale; kelvin does not, and taking the log of a
# temperature throws away the calibration the file was written with.
if calibrated:
    spectra_display = spectra
    display_label = 'Antenna temperature (K)'
else:
    spectra_display = 10 * np.log10(np.maximum(spectra, 1e-30))
    display_label = 'Power (dB)'

title = make_title()
print(f'Plot title: {title}')
print(f'Plotting in: {display_label}')

## 5. Waterfall plot

How the spectrum evolves with time; each row is one integration. Calibrated data are
shown directly in kelvin; uncalibrated counts are converted to dB for display only.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Convert timestamps to minutes from start
t_minutes = (timestamps - timestamps[0]) / 60.0

# Robust colour limits from the data
vmin = np.percentile(spectra_display, 5)
vmax = np.percentile(spectra_display, 95)

im = ax.imshow(
    spectra_display,
    aspect='auto',
    origin='lower',
    extent=[freq_mhz[0], freq_mhz[-1], t_minutes[0], t_minutes[-1]],
    cmap='viridis',
    vmin=vmin,
    vmax=vmax,
)

cb = fig.colorbar(im, ax=ax, label=display_label)
ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Time (minutes from start)')
ax.set_title(f'Waterfall — {title}')

# Mark the H I rest frequency
ax.axvline(x=1420.405, color='r', linestyle='--', linewidth=0.8, alpha=0.7, label='H I 1420.405 MHz')
ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## 6. Mean spectrum

Average all integrations to produce a single high-SNR spectrum. Averaging happens on the
spectra themselves — kelvin or counts, both linear in power — and never on the dB, since
the mean of a set of logarithms is not the logarithm of their mean.

For a **drift** scan the rows are different pieces of sky, so a straight mean is a
convolution with the transit, not a deeper integration on one source. Check
`observation_mode` above before reading too much into this plot.

In [ ]:
# Average the spectra themselves, never the dB - a mean of logarithms is not
# the logarithm of the mean, and the difference is a bias that grows with the
# noise. Kelvin and counts are both linear in power, so both average directly.
mean_spectrum = np.mean(spectra, axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(freq_mhz, mean_spectrum, linewidth=0.5, color='steelblue')
ax.axvline(x=1420.405, color='r', linestyle='--', linewidth=0.8, alpha=0.7, label='H I 1420.405 MHz')

ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel(f'Antenna temperature ({units})' if calibrated else 'Power (counts)')
ax.set_title(f'Mean Spectrum ({n_spectra} integrations) — {title}')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)

# Annotate with key metadata
info_text = f"SDR: {attrs.get('sdr_type', '?')}  |  Gain: {attrs.get('gain_db', '?')} dB"
info_text += f"  |  BW: {attrs.get('sample_rate_hz', 0)/1e6:.1f} MHz  |  {n_channels} ch"
total_integration = np.sum(integration_times)
info_text += f"  |  Total integration: {total_integration:.1f}s"
ax.text(0.01, 0.02, info_text, transform=ax.transAxes, fontsize=7,
        color='gray', verticalalignment='bottom')

plt.tight_layout()
plt.show()

## 7. Zoomed view around the hydrogen line

Zoom into a narrow window around 1420.405 MHz to see the H I emission detail.

In [ ]:
# Zoom to +/- 0.5 MHz around H I
h1_freq = 1420.405
zoom_width = 0.5  # MHz either side
mask = (freq_mhz >= h1_freq - zoom_width) & (freq_mhz <= h1_freq + zoom_width)

if mask.sum() > 0:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                                    gridspec_kw={'height_ratios': [1, 2]})

    ax1.plot(freq_mhz[mask], mean_spectrum[mask], linewidth=0.8, color='steelblue')
    ax1.axvline(x=h1_freq, color='r', linestyle='--', linewidth=0.8, alpha=0.7)
    ax1.set_ylabel(f'Antenna temperature ({units})' if calibrated else 'Power (counts)')
    ax1.set_title(f'H I Region Detail — {title}')
    ax1.grid(True, alpha=0.3)

    zoomed_spectra = spectra_display[:, mask]
    zoomed_freq = freq_mhz[mask]
    vmin_z = np.percentile(zoomed_spectra, 5)
    vmax_z = np.percentile(zoomed_spectra, 95)

    im = ax2.imshow(
        zoomed_spectra,
        aspect='auto',
        origin='lower',
        extent=[zoomed_freq[0], zoomed_freq[-1], t_minutes[0], t_minutes[-1]],
        cmap='viridis',
        vmin=vmin_z,
        vmax=vmax_z,
    )
    fig.colorbar(im, ax=ax2, label=display_label)
    ax2.axvline(x=h1_freq, color='r', linestyle='--', linewidth=0.8, alpha=0.7)
    ax2.set_xlabel('Frequency (MHz)')
    ax2.set_ylabel('Time (minutes from start)')

    plt.tight_layout()
    plt.show()
else:
    print(f'H I line at {h1_freq} MHz is outside the observed band ({freq_mhz[0]:.3f} - {freq_mhz[-1]:.3f} MHz)')

In [ ]:
hf.close()